In [ ]:

import json
import time

import requests

BASE_URL = "http://localhost:8000"
OWNER_ID = "test-user-001"


def print_response(method: str, url: str, resp: requests.Response) -> None:
    print(f"\n{'_'*60}")
    print(f"{resp.status_code}: {method} {url}")
    try:
        print(f"{json.dumps(resp.json(), indent=2, default=str)}")
    except Exception:
        print(f"{resp.text}")
    print(f"{'_'*60}")



In [ ]:
print("# 1. HEALTH CHECK")
print("#"*60)

resp = requests.get(f"{BASE_URL}/health")
print_response("GET", f"{BASE_URL}/health", resp)


In [ ]:
print("# 2. CREATE KEY - production")
print("#"*60)

payload = {
    "owner_id": OWNER_ID,
    "name": "Test Key Alpha",
    "description": "Created by t.py",
    "environment": "production",
    "scopes": ["read", "write", "admin"],
    "expires_at": "2099-12-31T23:59:59Z",
}
resp = requests.post(f"{BASE_URL}/v1/keys", json=payload)
print_response("POST", f"{BASE_URL}/v1/keys", resp)

key_data = resp.json()
key_id = key_data.get("id")
raw_key = key_data.get("raw_key")
print(f"\n>>> Saved key_id = {key_id}")
print(f">>> Saved raw_key = {raw_key}")


In [ ]:
print("# 3. CREATE KEY - staging")
print("#"*60)

payload = {
    "owner_id": OWNER_ID,
    "name": "Test Key Staging",
    "environment": "staging",
    "scopes": ["read"],
}
resp = requests.post(f"{BASE_URL}/v1/keys", json=payload)
print_response("POST", f"{BASE_URL}/v1/keys", resp)

staging_key_data = resp.json()
staging_key_id = staging_key_data.get("id")
staging_raw_key = staging_key_data.get("raw_key")
print(f"\n>>> Saved key_id = {staging_key_id}")
print(f">>> Saved raw_key = {staging_raw_key}")

In [ ]:
print("# 4. CREATE KEY - development")
print("#"*60)

payload = {
    "owner_id": OWNER_ID,
    "name": "Test Key Dev",
    "environment": "development",
    "scopes": [],
}
resp = requests.post(f"{BASE_URL}/v1/keys", json=payload)
print_response("POST", f"{BASE_URL}/v1/keys", resp)

dev_key_data = resp.json()
dev_key_id = dev_key_data.get("id")
dev_raw_key = dev_key_data.get("raw_key")
print(f"\n>>> Saved key_id = {dev_key_id}")
print(f">>> Saved raw_key = {dev_raw_key}")

In [ ]:
print("# 5. LIST KEYS - no filter")
print("#"*60)

resp = requests.get(
    f"{BASE_URL}/v1/keys",
    params={"owner_id": OWNER_ID, "limit": 10, "offset": 0},
)
print_response("GET", f"{BASE_URL}/v1/keys?owner_id={OWNER_ID}&limit=10&offset=0", resp)


In [ ]:
print("# 6. LIST KEYS - filter status=active")
print("#"*60)

resp = requests.get(
    f"{BASE_URL}/v1/keys",
    params={"owner_id": OWNER_ID, "status": "active", "limit": 10, "offset": 0},
)
print_response("GET", f"{BASE_URL}/v1/keys?owner_id={OWNER_ID}&status=active&limit=10&offset=0", resp)


In [ ]:
print("# 7. LIST KEYS - pagination offset=2")
print("#"*60)

resp = requests.get(
    f"{BASE_URL}/v1/keys",
    params={"owner_id": OWNER_ID, "limit": 10, "offset": 2},
)
print_response("GET", f"{BASE_URL}/v1/keys?owner_id={OWNER_ID}&limit=10&offset=2", resp)


In [ ]:
print("# 8. GET SINGLE KEY")
print("#"*60)

resp = requests.get(f"{BASE_URL}/v1/keys/{key_id}")
print_response("GET", f"{BASE_URL}/v1/keys/{key_id}", resp)


In [ ]:

print("# 9. GET NON-EXISTENT KEY (404)")
print("#"*60)

fake_id = "00000000-0000-0000-0000-000000000000"
resp = requests.get(f"{BASE_URL}/v1/keys/{fake_id}")
print_response("GET", f"{BASE_URL}/v1/keys/{fake_id}", resp)

In [ ]:

print("# 10. UPDATE KEY")
print("#"*60)

payload = {
    "name": "Updated Key Alpha",
    "description": "Updated by t.py",
    "scopes": ["read"],
}
resp = requests.patch(f"{BASE_URL}/v1/keys/{key_id}", json=payload)
print_response("PATCH", f"{BASE_URL}/v1/keys/{key_id}", resp)

In [ ]:

print("# 11. UPDATE NON-EXISTENT KEY (404)")
print("#"*60)

resp = requests.patch(f"{BASE_URL}/v1/keys/{fake_id}", json={"name": "nope"})
print_response("PATCH", f"{BASE_URL}/v1/keys/{fake_id}", resp)

In [ ]:

print("# 12. VERIFY KEY - valid")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": raw_key},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:

print("# 13. VERIFY KEY - invalid/garbage")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": "sk_live_thisisafakekey123456789"},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:

print("# 14. VERIFY KEY - validation error (too short)")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": "short"},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:

print("# 15. VERIFY STAGING KEY")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": staging_raw_key},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:

print("# 16. VERIFY DEV KEY")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": dev_raw_key},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:

print("# 17. GET USAGE STATS")
print("#"*60)

resp = requests.get(f"{BASE_URL}/v1/keys/{key_id}/usage")
print_response("GET", f"{BASE_URL}/v1/keys/{key_id}/usage", resp)

In [ ]:

print("# 18. GET USAGE STATS - non-existent key (404)")
print("#"*60)

resp = requests.get(f"{BASE_URL}/v1/keys/{fake_id}/usage")
print_response("GET", f"{BASE_URL}/v1/keys/{fake_id}/usage", resp)

In [ ]:

print("# 19. ROTATE KEY")
print("#"*60)

resp = requests.post(f"{BASE_URL}/v1/keys/{key_id}/rotate")
print_response("POST", f"{BASE_URL}/v1/keys/{key_id}/rotate", resp)

rotated_data = resp.json()
new_raw_key = rotated_data.get("raw_key")
print(f"\n>>> New raw_key after rotation = {new_raw_key}")

In [ ]:
print("# 20. VERIFY OLD KEY - after rotation (should fail)")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": raw_key},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:
print("# 21. VERIFY NEW KEY - after rotation (should succeed)")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": new_raw_key},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:
print("# 22. GET USAGE STATS - after rotation")
print("#"*60)

resp = requests.get(f"{BASE_URL}/v1/keys/{key_id}/usage")
print_response("GET", f"{BASE_URL}/v1/keys/{key_id}/usage", resp)

In [ ]:
print("# 23. REVOKE KEY")
print("#"*60)

resp = requests.post(f"{BASE_URL}/v1/keys/{key_id}/revoke")
print_response("POST", f"{BASE_URL}/v1/keys/{key_id}/revoke", resp)

In [ ]:
print("# 24. VERIFY REVOKED KEY (should fail)")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": new_raw_key},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:
print("# 25. LIST KEYS - filter status=revoked")
print("#"*60)

resp = requests.get(
    f"{BASE_URL}/v1/keys",
    params={"owner_id": OWNER_ID, "status": "revoked", "limit": 10, "offset": 0},
)
print_response("GET", f"{BASE_URL}/v1/keys?owner_id={OWNER_ID}&status=revoked&limit=10&offset=0", resp)


In [ ]:
print("# 26. REVOKE NON-EXISTENT KEY (404)")
print("#"*60)

resp = requests.post(f"{BASE_URL}/v1/keys/{fake_id}/revoke")
print_response("POST", f"{BASE_URL}/v1/keys/{fake_id}/revoke", resp)

In [ ]:
print("# 27. ROTATE NON-EXISTENT KEY (404)")
print("#"*60)

resp = requests.post(f"{BASE_URL}/v1/keys/{fake_id}/rotate")
print_response("POST", f"{BASE_URL}/v1/keys/{fake_id}/rotate", resp)

In [ ]:
print("# 28. DELETE STAGING KEY")
print("#"*60)

resp = requests.delete(f"{BASE_URL}/v1/keys/{staging_key_id}")
print_response("DELETE", f"{BASE_URL}/v1/keys/{staging_key_id}", resp)

In [ ]:
print("# 29. GET DELETED KEY (404)")
print("#"*60)

resp = requests.get(f"{BASE_URL}/v1/keys/{staging_key_id}")
print_response("GET", f"{BASE_URL}/v1/keys/{staging_key_id}", resp)

In [ ]:
print("# 30. DELETE NON-EXISTENT KEY (404)")
print("#"*60)

resp = requests.delete(f"{BASE_URL}/v1/keys/{fake_id}")
print_response("DELETE", f"{BASE_URL}/v1/keys/{fake_id}", resp)

In [ ]:
print("# 31. DELETE DEV KEY")
print("#"*60)

resp = requests.delete(f"{BASE_URL}/v1/keys/{dev_key_id}")
print_response("DELETE", f"{BASE_URL}/v1/keys/{dev_key_id}", resp)

In [ ]:
print("# 32. CREATE KEY - minimal payload")
print("#"*60)

payload = {
    "owner_id": OWNER_ID,
    "name": "Minimal Key",
}
resp = requests.post(f"{BASE_URL}/v1/keys", json=payload)
print_response("POST", f"{BASE_URL}/v1/keys", resp)

minimal_key_data = resp.json()
minimal_key_id = minimal_key_data.get("id")
minimal_raw_key = minimal_key_data.get("raw_key")

In [ ]:
print("# 33. VERIFY MINIMAL KEY")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys/verify",
    json={"key": minimal_raw_key},
)
print_response("POST", f"{BASE_URL}/v1/keys/verify", resp)

In [ ]:
print("# 34. CREATE KEY - missing owner_id (422)")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys",
    json={"name": "No Owner"},
)
print_response("POST", f"{BASE_URL}/v1/keys", resp)


print("\n" + "#"*60)
print("# 35. CREATE KEY - invalid environment (422)")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys",
    json={"owner_id": OWNER_ID, "name": "Bad Env", "environment": "invalid_env"},
)
print_response("POST", f"{BASE_URL}/v1/keys", resp)


In [ ]:
print("# 36. CREATE KEY - empty name (422)")
print("#"*60)

resp = requests.post(
    f"{BASE_URL}/v1/keys",
    json={"owner_id": OWNER_ID, "name": ""},
)
print_response("POST", f"{BASE_URL}/v1/keys", resp)

In [ ]:
print("# 37. GET USAGE STATS - custom limit=5")
print("#"*60)

resp = requests.get(f"{BASE_URL}/v1/keys/{key_id}/usage?limit=5")
print_response("GET", f"{BASE_URL}/v1/keys/{key_id}/usage?limit=5", resp)


In [ ]:
print("# 38. LIST KEYS - different owner (empty)")
print("#"*60)

resp = requests.get(
    f"{BASE_URL}/v1/keys",
    params={"owner_id": "nonexistent-owner-999", "limit": 10, "offset": 0},
)
print_response("GET", f"{BASE_URL}/v1/keys?owner_id=nonexistent-owner-999&limit=10&offset=0", resp)



In [ ]:
print("# 39. CLEANUP - delete minimal key")
print("#"*60)

resp = requests.delete(f"{BASE_URL}/v1/keys/{minimal_key_id}")
print_response("DELETE", f"{BASE_URL}/v1/keys/{minimal_key_id}", resp)


In [ ]:
print("# 40. CLEANUP - delete revoked key")
print("#"*60)

resp = requests.delete(f"{BASE_URL}/v1/keys/{key_id}")
print_response("DELETE", f"{BASE_URL}/v1/keys/{key_id}", resp)
